# B2.10 · Evaluating a security harness

**Function B — Product & Application Security → The Security Automation / Harness Engineer**  ·  *AI for Security*

---

**Risk.** A single successful run hides how unreliable an agent really is; a hallucinated finding looks exactly like a real one.

**Control.** Ground truth + blind protocol + execution-verified scoring; report reliability (pass^k) not just capability (pass@k); never quote schema conformance as accuracy.

**This lab.** Evaluate a security harness properly: four stages, two numbers, and the collision bug that quietly randomises everyone else's results.

| | |
|---|---|
| Open-source tooling | Cyber Commons eval harness, Checkov, CyberGym, Inspect |
| Open-weight models | Llama 3.3, GLM-4.6, Kimi K2 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("B2.10"))

This is the flagship harness lab and the anchor for the whole eval story. The full version with real corpora lives in `labs/b2.10-eval-harness`; this notebook is the same design, offline, in a form you can read end to end.

**Stage 1 — ingest.** Conformance is decided here, and it is structural.

In [ ]:
from cybercommons import evalkit

truths = {
 "q1": evalkit.Truth("q1", "CWE-89", "CWE-89/1.py"),
 "q2": evalkit.Truth("q2", "CWE-78", "CWE-78/1.py"),
 "q3": evalkit.Truth("q3", "CWE-22", "CWE-22/3.c"),
 "q4": evalkit.Truth("q4", "CWE-798", "CWE-798/2.py"),
}
answers = {
 "q1": '{"qid":"q1","cwe":"CWE-89","file":"CWE-89/1.py","line":2,'
       '"rationale":"user input concatenated into the query string"}',
 "q2": '{"qid":"q2","cwe":"CWE-89","file":"CWE-78/1.py","line":3,'
       '"rationale":"untrusted input reaches a shell"}',
 "q3": '{"qid":"q3","cwe":"CWE-22","file":"CWE-89/1.py","line":1,'
       '"rationale":"path built from user input"}',
 "q4": 'I think this file has a hardcoded credential.',
}
for qid, raw in answers.items():
    ans, note = evalkit.Answer.parse(raw)
    print(f"{qid}: {note}")

**Stage 2 — path matching.** This single line is the difference between a benchmark and a lottery.

In [ ]:
print("parent-dir + filename (correct):")
print("  ", evalkit.path_key("CWE-89/1.py"), "vs", evalkit.path_key("CWE-79/1.py"),
      "→ distinct")
print("bare basename (the bug):")
print("  ", "1.py", "vs", "1.py", "→ identical; q3's wrong answer would score as right")

**Stages 3 and 4 — expert proxy and dual judges.**

In [ ]:
rep = evalkit.evaluate(answers, truths)
print(rep.render())
print("\nfailures:")
for f in rep.failures:
    print(f"  {f['qid']}  [{f['stage']}]  {f['why']}")

Read the two headline numbers together. Conformance is 0.75 only because one answer was prose; with structured output it would be 1.00 and would say nothing at all about quality. Expert accuracy is the number that means something.

### Expect

q1–q3 conform and q4 does not. Conformance is 0.75 while expert accuracy is 0.375 — q1 scores 1.0, q2 scores 0.5 (right file, wrong class), q3 and q4 score 0. The failures list names the reason per question.

### Your turn

Change q3's file to `CWE-22/3.c` and re-run. Then deliberately break `path_key` to use the bare basename and re-run again: watch an accuracy number improve for no reason at all.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/B2.10.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*